In [1]:
import psycopg2
from datetime import datetime, timedelta
import time

# Configuración de conexión
conn = psycopg2.connect(
    user="postgres",
    password="postgres",
    host="127.0.0.1",
    port="5432",
    database="postgres"
)
cursor = conn.cursor()

# Rango de fechas
start_date = datetime.strptime("2022-12-31", "%Y-%m-%d")
end_date = datetime.strptime("2023-01-30", "%Y-%m-%d")
current_date = start_date

# Script base
query_template = """
INSERT INTO hosts_cumulated
WITH

yesterday AS (
SELECT
	*
FROM hosts_cumulated
WHERE 1=1
	AND date = DATE('{yesterday}')
),

today AS (
SELECT
	host,
	DATE(event_time) AS date_active
FROM events e 
WHERE 1=1
	AND DATE(e.event_time) = DATE('{today}')
	AND e.host IS NOT NULL
GROUP BY 1, 2
),


results AS (
SELECT
	COALESCE(t.host, y.host) AS host,
	CASE
		WHEN y.dates_active IS NULL THEN ARRAY[t.date_active]
		WHEN t.date_active IS NULL THEN y.dates_active
		ELSE ARRAY[t.date_active] || y.dates_active
	END AS dates_active,
	COALESCE(t.date_active, y.date + INTERVAL '1 day') AS date 
FROM today t
FULL OUTER JOIN yesterday y
ON 1=1
	AND t.host = y.host
)
	
SELECT * FROM results;
"""

# Ejecutar el script para cada día
while current_date <= end_date:
    yesterday = current_date.strftime("%Y-%m-%d")
    today = (current_date + timedelta(days=1)).strftime("%Y-%m-%d")
    query = query_template.format(yesterday=yesterday, today=today)

    try:
        cursor.execute(query)
        conn.commit()
        print(f"Query ejecutada para fecha: {yesterday} -> {today}")
        time.sleep(1)
    except Exception as e:
        print(f"Error ejecutando query para fecha: {yesterday} -> {today}: {e}")
        conn.rollback()
    
    current_date += timedelta(days=1)

# Cerrar conexión
cursor.close()
conn.close()


Query ejecutada para fecha: 2022-12-31 -> 2023-01-01
Query ejecutada para fecha: 2023-01-01 -> 2023-01-02
Query ejecutada para fecha: 2023-01-02 -> 2023-01-03
Query ejecutada para fecha: 2023-01-03 -> 2023-01-04
Query ejecutada para fecha: 2023-01-04 -> 2023-01-05
Query ejecutada para fecha: 2023-01-05 -> 2023-01-06
Query ejecutada para fecha: 2023-01-06 -> 2023-01-07
Query ejecutada para fecha: 2023-01-07 -> 2023-01-08
Query ejecutada para fecha: 2023-01-08 -> 2023-01-09
Query ejecutada para fecha: 2023-01-09 -> 2023-01-10
Query ejecutada para fecha: 2023-01-10 -> 2023-01-11
Query ejecutada para fecha: 2023-01-11 -> 2023-01-12
Query ejecutada para fecha: 2023-01-12 -> 2023-01-13
Query ejecutada para fecha: 2023-01-13 -> 2023-01-14
Query ejecutada para fecha: 2023-01-14 -> 2023-01-15
Query ejecutada para fecha: 2023-01-15 -> 2023-01-16
Query ejecutada para fecha: 2023-01-16 -> 2023-01-17
Query ejecutada para fecha: 2023-01-17 -> 2023-01-18
Query ejecutada para fecha: 2023-01-18 -> 2023